# MindClash – Reglas de Asociación
## 1. Presentación del problema

**Patrones de aprendizaje y rendimiento en la aplicación MindClash**

MindClash es una aplicación desarrollada por estudiantes de Ingeniería en Ciencias Computacionales de CETYS Universidad, enfocada en el aprendizaje gamificado y competitivo de temas relacionados con la programación, algoritmos, bases de datos, inteligencia artificial y seguridad informática.
A través de duelos, quizzes diarios y retos por niveles, los usuarios pueden medir su conocimiento, ganar puntos de experiencia (XP) y comparar su progreso en un leaderboard global.

El presente proyecto aplica técnicas de minería de datos, particularmente reglas de asociación mediante el algoritmo Apriori, para descubrir patrones en el comportamiento de aprendizaje de los usuarios.
El objetivo es identificar relaciones entre los temas dominados por los estudiantes y su desempeño general, encontrando asociaciones entre conceptos que tienden a ser comprendidos de forma conjunta o secuencial.

Con base en estos patrones, se busca:
* Comprender las fortalezas y debilidades temáticas de los usuarios.
* Proponer rutas personalizadas de aprendizaje que potencien su progreso dentro de MindClash.
* Explorar cómo las habilidades en ciertas áreas (por ejemplo, programación o estructuras de datos) se relacionan con el dominio de otros campos (como bases de datos o seguridad).

En resumen, este análisis permitirá transformar los datos de interacción y rendimiento de los estudiantes en conocimiento útil, favoreciendo una experiencia educativa más adaptativa e inteligente dentro del ecosistema de FoxCoding y MindClash.


## 2. Recolección de datos

### Fuente de los datos

**Origen:**
Los datos utilizados en este proyecto corresponden a información simulada basada en el funcionamiento real de la **aplicación MindClash**.
Si bien este conjunto es **mock data** (datos de ejemplo generados con propósitos de análisis y demostración), refleja fielmente la estructura, las variables y los patrones que podrían extraerse en el futuro a partir de la base de datos real de usuarios de la aplicación.

En versiones futuras, los datos podrán obtenerse directamente desde la infraestructura de Firebase/Firestore utilizada por MindClash, donde se almacenan los resultados de los quizzes, duelos 1v1, progreso de los usuarios y métricas de aprendizaje.

---

### Tipo de archivo

Los datos fueron estructurados en un archivo **CSV (Comma-Separated Values)** llamado `attempts.csv.`
Este formato permite un fácil manejo en librerías de análisis como Pandas y es ampliamente compatible con herramientas de minería de datos como **RapidMiner**.

En un entorno de producción, los datos podrían exportarse automáticamente desde Firestore mediante funciones de backend que recojan las métricas más relevantes de cada usuario.

---

### Descripción de las columnas

Cada registro en el conjunto representa una respuesta individual de un usuario a una pregunta dentro de un quiz o duelo.
Es decir, cada fila corresponde a un intento de respuesta, lo que permite analizar el comportamiento de los estudiantes a un nivel muy granular.

---

### Columnas del dataset

| Columna        | Tipo                | Descripción                                                                                  |
| -------------- | ------------------- | -------------------------------------------------------------------------------------------- |
| **uid**        | Categórica          | Identificador único del usuario.                                                             |
| **quizId**     | Categórica          | Identificador del quiz en el que se realizó la pregunta (por ejemplo, *quiz_1*, *quiz_2*).   |
| **questionId** | Categórica          | Identificador único de la pregunta dentro del quiz.                                          |
| **subject**    | Categórica          | Materia o área de conocimiento relacionada (Programación, Algoritmos, Bases de Datos, etc.). |
| **concepts**   | Categórica múltiple | Conceptos o temas específicos asociados a la pregunta (por ejemplo: *arrays;concurrency*).   |
| **difficulty** | Ordinal             | Nivel de dificultad asignado a la pregunta (Easy, Medium, Hard).                             |
| **isCorrect**  | Binaria             | Indica si la respuesta fue correcta (1) o incorrecta (0).                                    |
| **responseMs** | Numérica            | Tiempo de respuesta en milisegundos.                                                         |
| **ts**         | Temporal            | Marca de tiempo (timestamp) en que se registró la respuesta.                                 |


---

### Naturaleza y relevancia de los datos

Este tipo de información representa la huella digital del aprendizaje dentro de MindClash.
Al analizar las combinaciones entre conceptos, dificultades y aciertos, es posible identificar patrones de dominio temático, debilidades recurrentes, y relaciones entre materias que los usuarios dominan simultáneamente.

Por ejemplo:

Si un usuario domina “arrays” y “concurrency”, existe alta probabilidad de que también comprenda “normalization” (tema de bases de datos).

Los usuarios con buenos tiempos de respuesta en preguntas difíciles de seguridad tienden también a acertar en preguntas de redes.

---

### Nota sobre el preprocesamiento

Para aplicar correctamente el algoritmo Apriori, se realizó un preprocesamiento que incluye:

* Conversión a formato transaccional: cada intento se transforma en una transacción con los items asociados (conceptos, dificultad, acierto, materia).

* Codificación de variables categóricas: los campos como difficulty o subject se representaron como valores discretos.

* Filtrado de ruido: se eliminaron registros con valores nulos o inconsistentes (por ejemplo, tiempos negativos o preguntas sin tema).

En el futuro, cuando la base de datos real esté disponible, estos mismos pasos podrán aplicarse sobre los registros obtenidos directamente desde Firestore mediante consultas o funciones Cloud.


In [2]:
import pandas as pd

# attempts: uid, subject, concepts(list), isCorrect
attempts = pd.read_csv("attempts.csv")

attempts.head()

,uid,quizId,questionId,subject,concepts,difficulty,isCorrect,responseMs,ts
0,u4,quiz_5,q1343,Networking,rest;tcp,Medium,0,13142,2025-10-08T11:13:59Z
1,u1,quiz_1,q1328,Databases,normalization,Easy,1,11416,2025-10-08T11:15:59Z
2,u1,quiz_5,q1327,Programming,arrays;concurrency,Easy,1,10865,2025-10-08T11:28:59Z
3,u5,quiz_2,q1347,Databases,sql-agg;indexing,Hard,0,24201,2025-10-08T11:59:59Z
4,u3,quiz_1,q1338,Security,csrf;auth,Hard,0,19249,2025-10-08T13:20:59Z


## 3. Preparación de los datos
Describe las transformaciones aplicadas:
- Limpieza (valores nulos, duplicados)
- Selección de columnas relevantes
- Conversión al formato de transacciones

## 4. Aplicación de reglas de asociación

### Algoritmo seleccionado: **Apriori**

El algoritmo **Apriori** fue seleccionado para identificar relaciones entre conceptos y materias dentro del proceso de aprendizaje de los usuarios de MindClash.
A través de este enfoque, se buscó descubrir patrones de dominio conjunto, es decir, qué temas tienden a ser comprendidos o dominados simultáneamente por los estudiantes con base en sus resultados dentro de los quizzes.

Este tipo de análisis permite revelar asociaciones pedagógicas naturales, como por ejemplo:

`Los usuarios que dominan “arrays” y “concurrency” también suelen acertar preguntas de “normalization”.`

---

### ¿Qué es el algoritmo Apriori?

**Apriori** es un algoritmo clásico de minería de datos empleado para descubrir conjuntos de ítems frecuentes en una base de datos transaccional y, a partir de ellos, derivar reglas de asociación que describen relaciones relevantes entre los elementos.

Su funcionamiento se basa en el principio de conocimiento a priori:

``Si un conjunto de ítems es frecuente, entonces todos sus subconjuntos también lo son.``

En el contexto de MindClash, esto significa que:

``Si un grupo de usuarios tiende a responder correctamente preguntas de ciertos temas (A), existe una alta probabilidad de que también respondan correctamente temas relacionados (B).``

Ejemplo:

``Si un estudiante domina los conceptos de “Recursion” y “Dynamic Programming”, es probable que también domine “Optimization”.``

---

### Funcionamiento general del algoritmo

1. Conversión de datos a transacciones:
Cada usuario se representó como una transacción, conteniendo los conceptos de las preguntas que ha respondido correctamente.

2. Cálculo del soporte mínimo:
Se determinaron los conjuntos de ítems (conceptos) que aparecen con una frecuencia mínima dentro de las transacciones de usuarios.

3. Generación de conjuntos frecuentes:
Se construyeron combinaciones crecientes de ítems frecuentes (pares, tríos, etc.), eliminando aquellos que no cumplían el umbral de soporte establecido.

4. Extracción de reglas de asociación:
A partir de los conjuntos frecuentes, se generaron reglas del tipo A → B, donde:

    * A (antecedente): conjunto de conceptos ya dominados.

    * B (consecuente): concepto que tiende a acompañar al grupo A.

    * Se calcularon las métricas soporte, confianza y lift para evaluar la relevancia de cada relación.

---

### Parámetros seleccionados

Los parámetros del modelo se ajustaron al tamaño del conjunto (23 transacciones) y al objetivo de obtener reglas interpretables sobre aprendizaje:

| Parámetro       | Valor   | Descripción                                                                                                        |
| --------------- | ------- | ------------------------------------------------------------------------------------------------------------------ |
| **min_support** | `0.08`  | Requiere que una combinación aparezca al menos en el 8% de los usuarios (≈ 2 estudiantes).                         |
| **confidence**  | `≥ 0.7` | Probabilidad de que el consecuente ocurra dado el antecedente. Asegura coherencia en las reglas.                   |
| **lift**        | `≥ 1.3` | Evalúa la fuerza real de la relación. Valores >1.0 indican una relación positiva; >1.3 una relación significativa. |
| **max_len**     | `3`     | Limita el tamaño de los conjuntos para mantener claridad interpretativa.                                           |


In [1]:
from mlxtend.frequent_patterns import apriori, association_rules


# 1) concept mastery by user (last 90 days)
cut = pd.Timestamp.utcnow() - pd.Timedelta(days=90)
recent = attempts[pd.to_datetime(attempts['ts']) >= cut]

# explode concepts
expl = recent.explode('concepts')

# compute correctness per (uid, concept)
agg = (expl.groupby(['uid','concepts'])['isCorrect']
           .mean()
           .reset_index(name='acc'))

# define mastery threshold
mastery = agg.assign(mastered = agg['acc'] >= 0.8)
mastered = mastery[mastery['mastered']].drop(columns=['acc','mastered'])

# transaction matrix (users × concepts mastered)
basket = (mastered
          .assign(val=1)
          .pivot_table(index='uid', columns='concepts', values='val', fill_value=0))

# 2) frequent itemsets
freq = apriori(basket, min_support=0.05, use_colnames=True)  # tune support

# 3) rules
rules = association_rules(freq, metric="lift", min_threshold=1.1)  # tune
rules = rules.sort_values(['lift','confidence'], ascending=False)

# keep actionable rules (limit length, stability, etc.)
rules = rules[(rules['antecedents'].apply(len) <= 3) &
              (rules['consequents'].apply(len) == 1) &
              (rules['confidence'] >= 0.6) &
              (rules['support'] >= 0.03)]

rules.head(20)


C:\Users\emili\PyCharmMiscProject\.venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,(arrays;concurrency),(normalization),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
3,(normalization),(arrays;concurrency),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
4,(arrays;concurrency),(sorting),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
5,(sorting),(arrays;concurrency),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
6,(encryption;xss),(joins;normalization),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
7,(joins;normalization),(encryption;xss),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
8,(encryption;xss),(overfitting),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
9,(overfitting),(encryption;xss),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
10,(gradient-descent),(routing),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0
11,(routing),(gradient-descent),0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.0,1.0,1.0


## 5. Análisis de resultados

### 🔍 Relaciones interesantes encontradas

El análisis realizado mediante el algoritmo Apriori permitió identificar patrones de aprendizaje y afinidad conceptual entre los usuarios de la aplicación MindClash.
Las reglas obtenidas muestran asociaciones claras entre diferentes temas de programación, algoritmos, bases de datos y seguridad, revelando cómo los estudiantes tienden a dominar ciertos grupos de conceptos de manera conjunta.

Entre las relaciones más destacadas se encuentran:

* Los usuarios que dominan arrays y concurrency tienden también a:

    * Comprender temas de normalización en bases de datos.

    * Resolver correctamente ejercicios de ordenamiento (sorting), reflejando un avance lógico en la comprensión de estructuras de datos.

* Los usuarios con dominio en encryption y XSS:

    * Suelen dominar también joins y normalization, lo que sugiere una conexión entre la seguridad web y la estructura lógica de los datos.

    * Además, muestran afinidad con el tema de overfitting, indicando un perfil con razonamiento analítico y comprensión transversal entre seguridad e inteligencia artificial.

* Los estudiantes con alto desempeño en gradient descent (descenso de gradiente) presentan:

    * Alta probabilidad de dominar routing y transactions, relacionando conceptos de optimización matemática con procesos de comunicación y manejo de datos.

* Finalmente, los usuarios que comprenden hashing y protocolos de red (TCP, routing) muestran una fuerte tendencia a dominar temas de criptografía y seguridad, reforzando la interconexión natural entre ambos campos.

En general, las reglas reflejan un patrón coherente:

``Los estudiantes que destacan en áreas estructurales (como algoritmos y redes) tienden también a dominar temas de organización lógica (bases de datos) y protección de información (seguridad).``

### 📊 Significado de los valores de soporte, confianza y lift

* Soporte (support)
Representa la frecuencia relativa con la que una combinación de conceptos aparece entre los usuarios analizados.
En este análisis, un soporte de 0.1667 indica que la relación está presente en aproximadamente 1 de cada 6 usuarios, lo cual es relevante considerando el tamaño reducido de la muestra.

* Confianza (confidence)
Indica la probabilidad de que el consecuente ocurra cuando el antecedente está presente.
En este caso, las reglas tienen una confianza de 1.0, lo que implica que cada vez que el usuario domina A, también domina B dentro del conjunto observado.
Si bien esto refleja asociaciones fuertes, también puede deberse al tamaño limitado del dataset (conjunto simulado).

* Lift (lift)
Evalúa la fuerza real de la asociación entre los ítems.
Los valores obtenidos (hasta 6.0) indican que la probabilidad de que A y B ocurran juntos es seis veces mayor que si fueran eventos independientes.
Esto sugiere una relación pedagógicamente significativa: los temas asociados se refuerzan entre sí dentro del proceso de aprendizaje.

### 🧩 Reglas útiles para el contexto del aprendizaje en MindClash

En el contexto educativo de MindClash, las reglas más relevantes son aquellas que pueden guiar estrategias de recomendación de contenido, diseñar rutas de aprendizaje personalizadas o mejorar la estructura de los quizzes.

Entre ellas destacan:

1. Arrays + Concurrency → Normalization

    * Interpretación: los estudiantes que entienden estructuras de datos y paralelismo tienden a comprender la organización lógica de bases de datos.

    * Aplicación: el sistema podría recomendar temas de normalization a usuarios que ya han demostrado dominio en arrays y concurrency, reforzando el aprendizaje cruzado entre materias.

2. Encryption + XSS → Joins + Normalization

    * Interpretación: quienes dominan seguridad web suelen tener facilidad para entender relaciones y dependencias en bases de datos.

    * Aplicación: promover desafíos integradores que unan seguridad y modelado de datos, fortaleciendo la transferencia de conocimiento entre campos.

3. Gradient Descent → Routing

    * Interpretación: existe una relación entre la optimización matemática y el entendimiento de flujos de comunicación en redes.

    * Aplicación: incorporar minijuegos o módulos donde conceptos de IA se apliquen a simulaciones de redes o tráfico de datos.

4. Hashing + TCP → Criptografía y Seguridad

    * Interpretación: el conocimiento en estructuras de datos y redes favorece el aprendizaje en seguridad informática.

    * Aplicación: crear rutas adaptativas donde el sistema recomiende temas de criptografía tras dominar hashing o protocolos TCP/IP.

### 💡 Reflexión general

Los resultados confirman que los datos recolectados por MindClash pueden servir como base para construir un motor de recomendación inteligente que sugiera el siguiente tema óptimo para cada usuario.
Además, demuestran que el aprendizaje en ciencias computacionales no ocurre de forma aislada, sino que sigue trayectorias cognitivas interconectadas entre áreas de algoritmos, bases de datos, redes y seguridad.

``En el futuro, estas reglas podrían alimentar un modelo de aprendizaje adaptativo dentro de la app, capaz de ajustar la dificultad, los temas sugeridos y los retos de MindClash en función del perfil de dominio individual de cada estudiante.``